In [ ]:
import numpy as np
import numpy.typing as npt

import matplotlib.pyplot as plt

In [ ]:
def reach_out_until_mask_zero(
    point: npt.NDArray[np.float64], vector: npt.NDArray[np.float64], mask: npt.NDArray[np.bool_]
) -> npt.NDArray[np.float64]:
    current_point = point.copy()
    unit_vector = vector / np.linalg.norm(vector)
    while True:
        int_current_point = np.round(current_point).astype(int)
        if (
            int_current_point[0] <= 0
            or int_current_point[0] == mask.shape[0] - 1
            or int_current_point[1] <= 0
            or int_current_point[1] == mask.shape[1] - 1
        ):
            # reached edge, consider this to be the end point
            break
        if not mask[int_current_point[0], int_current_point[1]]:
            # reached a zero in the mask, this is the end point
            break
        current_point += unit_vector
    return current_point


def numpy_vectorised_reach_out_until_mask_zero(
    point: npt.NDArray[np.float64], vector: npt.NDArray[np.float64], mask: npt.NDArray[np.bool_]
) -> npt.NDArray[np.float64]:
    unit_vector = vector / np.linalg.norm(vector)
    # maximum possible distance is the diagonal of the mask.
    max_distance = np.linalg.norm(mask.shape)
    x0, y0 = point
    dx, dy = unit_vector
    # create array for the number of times we step in the direction of the vector
    steps = np.arange(0, max_distance)
    # create vector
    all_ys = (y0 + steps * dy).astype(int)
    all_xs = (x0 + steps * dx).astype(int)
    # valid points only - create a boolean array indicating allowed points by getting
    # rid of those that are out of bounds of the mask
    valid_points_bool_array = (all_xs > 0) & (all_xs < mask.shape[0] - 1) & (all_ys > 0) & (all_ys < mask.shape[1] - 1)
    # use the boolean array to cull the arrays
    valid_xs = all_xs[valid_points_bool_array]
    valid_ys = all_ys[valid_points_bool_array]
    # get the values from the mask using the valid points
    mask_values = mask[valid_xs, valid_ys]
    # get the zeros by inverting the mask values, so positives represent zeros in the mask
    zero_mask = ~mask_values
    if not np.any(zero_mask):
        # if there are no zeros, return the point at the edge of the mask in the direction of the vector
        return np.array(
            [
                x0 + dx * max_distance,
                y0 + dy * max_distance,
            ]
        ).astype(int)
    # argmax returns the first index of the maximum value, which for a bool array, is the first true value
    first_zero_index = np.argmax(zero_mask)
    return np.array(
        [
            valid_xs[first_zero_index],
            valid_ys[first_zero_index],
        ]
    ).astype(int)


mask = np.zeros((100, 100), dtype=bool)
mask[30:70, 30:70] = True

point = np.array([50, 50]).astype(np.float64)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(mask, cmap="gray")
ax.scatter(point[1], point[0], color="black", s=50)


vectors = [
    (np.array([1, 0]), np.array([70, 50])),
    (np.array([0, 1]), np.array([50, 70])),
    (np.array([-1, 0]), np.array([29, 50])),
    (np.array([0, -1]), np.array([50, 29])),
    (np.array([1, 1]), np.array([69.79898987, 69.79898987])),
    (np.array([-1, 1]), np.array([30.20101013, 69.79898987])),
    (np.array([-1, -1]), np.array([29.49390335, 29.49390335])),
    (np.array([1, -1]), np.array([69.79898987, 30.20101013])),
    (np.array([2, 1]), np.array([69.6773982, 59.8386991])),
    (np.array([1, 2]), np.array([59.8386991, 69.6773982])),
    (np.array([-2, 1]), np.array([29.42817461, 60.2859127])),
    (np.array([-1, 2]), np.array([40.1613009, 69.6773982])),
    (np.array([-2, -1]), np.array([29.42817461, 39.7140873])),
    (np.array([-1, -2]), np.array([39.7140873, 29.42817461])),
    (np.array([2, -1]), np.array([69.6773982, 40.1613009])),
    (np.array([1, -2]), np.array([60.2859127, 29.42817461])),
]


for vector, expected_endpoint in vectors:
    expected_rounded_endpoitn = np.round(expected_endpoint).astype(int)

    endpoint = numpy_vectorised_reach_out_until_mask_zero(point, vector, mask)

    plt.scatter(endpoint[1], endpoint[0], color="red", s=20)

    rounded_endpoint = np.round(endpoint).astype(int)

plt.show()

In [ ]:
%%timeit
for vector, expected_endpoint in vectors:
    expected_rounded_endpoitn = np.round(expected_endpoint).astype(int)
    endpoint = reach_out_until_mask_zero(point, vector, mask)

In [ ]:
%%timeit
for vector, expected_endpoint in vectors:
    expected_rounded_endpoitn = np.round(expected_endpoint).astype(int)
    endpoint = numpy_vectorised_reach_out_until_mask_zero(point, vector, mask)